In [9]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
import warnings

warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../../'))
sys.path.append(root_path)

In [10]:
from core.backtesting import BacktestingEngine

backtesting = BacktestingEngine(root_path=root_path, load_cached_data=True)

In [14]:
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
from controllers.directional_trading.macd_bb_v1 import MACDBBV1ControllerConfig
import datetime
from decimal import Decimal

# Controller configuration
connector_name = "binance_perpetual"
trading_pair = "ALPHA-USDT"
interval = "15m"
fast_ma = 21
slow_ma = 42
signal_ma = 9
bb_length = 100
bb_std = 1.0
total_amount_quote = 1000
max_executors_per_side = 2
take_profit = 0.04
stop_loss = 0.02
trailing_stop_activation_price = 0.007
trailing_stop_trailing_delta = 0.003
time_limit = 60 * 60 * 24 * 2
cooldown_time = 60 * 15

start = int(datetime.datetime(2024, 10, 25).timestamp())
end = int(datetime.datetime(2024, 11, 10).timestamp())


# Creating the instance of the configuration and the controller
config = MACDBBV1ControllerConfig(
    connector_name=connector_name,
    trading_pair=trading_pair,
    interval=interval,
    macd_fast=fast_ma,
    macd_slow=slow_ma,
    macd_signal=signal_ma,
    bb_length=bb_length,
    bb_std=bb_std,
    total_amount_quote=Decimal(total_amount_quote),
    take_profit=Decimal(take_profit),
    stop_loss=Decimal(stop_loss),
    trailing_stop=TrailingStop(activation_price=Decimal(trailing_stop_activation_price), trailing_delta=Decimal(trailing_stop_trailing_delta)),
    time_limit=time_limit,
    max_executors_per_side=max_executors_per_side,
    cooldown_time=cooldown_time,
    candles_connector=None,         # <--- Add this line
    candles_trading_pair=None       # <--- Add this line
)
config

MACDBBV1ControllerConfig(id=None, controller_name='macd_bb_v1', controller_type='directional_trading', total_amount_quote=Decimal('1000'), manual_kill_switch=False, candles_config=[], connector_name='binance_perpetual', trading_pair='ALPHA-USDT', max_executors_per_side=2, cooldown_time=900, leverage=20, position_mode='HEDGE', stop_loss=Decimal('0.0200000000000000004163336342344337026588618755340576171875'), take_profit=Decimal('0.040000000000000000832667268468867405317723751068115234375'), time_limit=172800, take_profit_order_type=<OrderType.LIMIT: 2>, trailing_stop=TrailingStop(activation_price=Decimal('0.007000000000000000145716771982051795930601656436920166015625'), trailing_delta=Decimal('0.003000000000000000062450045135165055398829281330108642578125')), candles_connector='binance_perpetual', candles_trading_pair='ALPHA-USDT', interval='15m', bb_length=100, bb_std=1.0, bb_long_threshold=0.0, bb_short_threshold=1.0, macd_fast=21, macd_slow=42, macd_signal=9)

In [ ]:
backtesting_result = await backtesting.run_backtesting(config, start, end, interval)

2025-05-23 07:36:45,008 - root - ERROR - Error getting last traded prices in connector <hummingbot.connector.derivative.binance_perpetual.binance_perpetual_derivative.BinancePerpetualDerivative object at 0x16961df30> for trading pairs ['ALPHA-USDT']: Cannot connect to host fapi.binance.com:443 ssl:default [nodename nor servname provided, or not known]


In [16]:
# Let's see what is inside the backtesting results
print(backtesting_result.get_results_summary())
backtesting_result.get_backtesting_figure()


Net PNL: $-49.49 (-4.95%) | Max Drawdown: $-128.55 (-12.99%)
Total Volume ($): 64000.00 | Sharpe Ratio: -0.43 | Profit Factor: 0.81
Total Executors: 64 | Accuracy Long: 0.60 | Accuracy Short: 0.52
Close Types: Take Profit: 1 | Stop Loss: 25 | Time Limit: 0 |
             Trailing Stop: 38 | Early Stop: 0



In [17]:
# 2. The executors dataframe: this is the dataframe that contains the information of the orders that were executed
import pandas as pd

executors_df = backtesting_result.executors_df
executors_df.head()

,id,timestamp,type,status,config,net_pnl_pct,net_pnl_quote,cum_fees_quote,filled_amount_quote,is_active,is_trading,custom_info,close_timestamp,close_type,controller_id,side
0,GetSrLBwUC96DzbNzpJnjEfLph6Zvms7ctrAs9YGmBrF,1729912500,position_executor,RunnableStatus.TERMINATED,{'id': 'GetSrLBwUC96DzbNzpJnjEfLph6Zvms7ctrAs9...,-0.0206053347559348003226187273639880004338920...,-10.302667377967400952343268727418035268783569...,0.29999999999999998889776975374843459576368331...,1000.0000000000001136868377216160297393798828125,False,False,"{'close_price': 0.07348, 'level_id': None, 'si...",1729940400,CloseType.STOP_LOSS,None,BUY
1,Do5VQyEsT92Tp7BMNtddq2yserVTiVxHZjk577WPFLk8,1729914300,position_executor,RunnableStatus.TERMINATED,{'id': 'Do5VQyEsT92Tp7BMNtddq2yserVTiVxHZjk577...,-0.0229920622150710622644265157532572629861533...,-11.496031107535531035068743221927434206008911...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 0.07291, 'level_id': None, 'si...",1729945800,CloseType.STOP_LOSS,None,BUY
2,DFdUgAQ2HdnSKJkqzNw3vLdUj2bubrE58trsq97EgBEA,1729961100,position_executor,RunnableStatus.TERMINATED,{'id': 'DFdUgAQ2HdnSKJkqzNw3vLdUj2bubrE58trsq9...,0.00770564784053167153854424498149455757811665...,3.85282392026583542232742729538585990667343139...,0.29999999999999993338661852249060757458209991...,999.9999999999998863131622783839702606201171875,False,False,"{'close_price': 0.07284, 'level_id': None, 'si...",1729968300,CloseType.TRAILING_STOP,None,BUY
3,DEuoWNKgnKBmhA8dHV5j8KFsscJEnBMwSS27P14AWe6z,1729962900,position_executor,RunnableStatus.TERMINATED,{'id': 'DEuoWNKgnKBmhA8dHV5j8KFsscJEnBMwSS27P1...,0.00826426592797762163411512403854430885985493...,4.13213296398881091420207667397335171699523925...,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 0.07284, 'level_id': None, 'si...",1729968300,CloseType.TRAILING_STOP,None,BUY
4,5mupeTCEs8JqF8GwwWYVM7U8yuyFhQJXquaXw6wU3spw,1730055600,position_executor,RunnableStatus.TERMINATED,{'id': '5mupeTCEs8JqF8GwwWYVM7U8yuyFhQJXquaXw6...,0.04257241379310332252128290519976872019469738...,21.2862068965516613161526038311421871185302734375,0.29999999999999998889776975374843459576368331...,1000,False,False,"{'close_price': 0.07563, 'level_id': None, 'si...",1730066400,CloseType.TAKE_PROFIT,None,BUY


### Backtesting Analysis

### Scatter of PNL per Trade
This bar chart illustrates the PNL for each individual trade. Positive PNLs are shown in green and negative PNLs in red, providing a clear view of profitable vs. unprofitable trades.


In [18]:
import plotly.express as px

# Create a new column for profitability
executors_df['profitable'] = executors_df['net_pnl_quote'] > 0

# Create the scatter plot
fig = px.scatter(
    executors_df,
    x="timestamp",
    y='net_pnl_quote',
    title='PNL per Trade',
    color='profitable',
    color_discrete_map={True: 'green', False: 'red'},
    labels={'timestamp': 'Timestamp', 'net_pnl_quote': 'Net PNL (Quote)'},
    hover_data=['filled_amount_quote', 'side']
)

# Customize the layout
fig.update_layout(
    xaxis_title="Timestamp",
    yaxis_title="Net PNL (Quote)",
    legend_title="Profitable",
    font=dict(size=12, color="white"),
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0.8)',  # Dark background
    paper_bgcolor='rgba(0,0,0,0.8)',  # Dark background for the entire plot area
    xaxis=dict(gridcolor="gray"),
    yaxis=dict(gridcolor="gray")
)

# Add a horizontal line at y=0 to clearly separate profits and losses
fig.add_hline(y=0, line_dash="dash", line_color="lightgray")

# Show the plot
fig.show()

### Histogram of PNL Distribution
The histogram displays the distribution of PNL values across all trades. It helps in understanding the frequency and range of profit and loss outcomes.


In [19]:
fig = px.histogram(executors_df, x='net_pnl_quote', title='PNL Distribution')
fig.show()


# Conclusion
We can see that the indicator has potential to bring good signals to trade and might be interesting to see how we can design a market maker that shifts the mid price based on this indicator.
A lot of the short signals are wrong but if we zoom in into the loss signals we can see that the losses are not that big and the wins are bigger and if we had implemented the trailing stop feature probably a lot of them are going to be profits.

# Next steps
- Filter only the loss signals and understand what you can do to prevent them
- Try different configuration values for the indicator
- Test in multiple markets, pick mature markets like BTC-USDT or ETH-USDT and also volatile markets like DOGE-USDT or SHIB-USDT